# Tester le RAG documentaire dans Google Colab

Ce notebook permet d'essayer le compagnon de l'article avec un PDF actuariel public de **30 pages**. Vous cliquez sur les boutons ▶ et écrivez vos questions. Aucune installation sur votre ordinateur.

**Avant le premier clic**

1. Connectez-vous à votre compte Google, puis cliquez sur **Copier sur Drive**. Vous travaillez dans votre copie personnelle ; vous ne modifiez pas le dépôt GitHub de l'auteur.
2. Dans **Exécution → Modifier le type d'exécution**, choisissez un **GPU** (T4 si proposé), puis enregistrez. Colab décide des ressources disponibles ; un GPU gratuit n'est pas garanti.
3. Exécutez les quatre cellules numérotées, l'une après l'autre. Attendez le message de réussite avant de passer à la suivante. Pour le premier essai, évitez **Tout exécuter**.

Le notebook utilise votre propre clé API OpenAI. La cellule 2 et chaque question effectuent des appels payants sur votre compte. Pour une question documentaire, le texte et les images des passages retenus sont envoyés à OpenAI. Le parsing et les embeddings tournent dans votre machine Colab.

**État de cette version** : parcours préparé et contrôlé hors ligne ; l'exécution complète sur un GPU Colab reste à confirmer. Un résultat de test simulé n'est pas une preuve de réussite sur ce PDF.


In [ ]:
# @title 1. Installer le compagnon
import importlib.util
from pathlib import Path
import subprocess

installation_terminee = False
if "session" in globals():
    session.close()
connexion_verifiee = False
document_pret = False
resultat = None

# Colab reçoit sa propre copie du code public, dans sa machine temporaire.
PROJET = Path("/content/docling-hybrid-rag")
DEPOT = "https://github.com/FranckTbn/docling-hybrid-rag.git"
REFERENCE = "codex/colab-reader"
if not PROJET.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REFERENCE, DEPOT, str(PROJET)], check=True)
elif not (PROJET / ".git").is_dir():
    raise RuntimeError("Le dossier de travail existe sans être le compagnon. Utilisez une nouvelle session Colab.")

origine = subprocess.check_output(["git", "-C", str(PROJET), "remote", "get-url", "origin"], text=True).strip()
if origine != DEPOT:
    raise RuntimeError("Le dossier appartient à un autre dépôt. Utilisez une nouvelle session Colab.")
REVISION = subprocess.check_output(["git", "-C", str(PROJET), "rev-parse", "HEAD"], text=True).strip()

# Seul ce client standard entre dans Colab ; les dépendances du RAG restent isolées.
chemin_client = PROJET / "colab_support.py"
if not chemin_client.is_file():
    raise RuntimeError("Cette révision ne contient pas encore le parcours Colab. Vérifiez le lien du README.")
specification = importlib.util.spec_from_file_location("rag_colab_client", chemin_client)
client = importlib.util.module_from_spec(specification)
specification.loader.exec_module(client)
PYTHON_RAG = client.prepare_environment(PROJET)
installation_terminee = True
print("Installation terminée. Passez à la cellule 2.")
print("Version du compagnon :", REVISION[:12])


## Ajouter votre clé, une seule fois

Ouvrez **Secrets** dans la barre de gauche (icône de clé). Ajoutez un secret nommé exactement **OPENAI_API_KEY**, collez votre clé dans sa valeur, puis activez **Accès au notebook**. Ne collez pas la clé dans une cellule.

Revenez ici et cliquez sur ▶. Le petit test de connexion évite de lancer un long parsing avec une clé inutilisable. Le modèle ci-dessous doit être accessible sur votre compte et accepter les images, l'API Responses, le raisonnement et les réponses JSON structurées. Vous pouvez conserver sa valeur pour commencer.


In [ ]:
# @title 2. Vérifier la connexion
modele = "gpt-5.6-luna" # @param {type:"string"}
connexion_verifiee = False
document_pret = False
resultat = None
if not globals().get("installation_terminee"):
    raise RuntimeError("Exécutez d'abord la cellule 1 jusqu’au message Installation terminée.")
from google.colab import userdata
from IPython.display import display, Markdown
import uuid
if "session" in globals():
    session.close()
try:
    cle = userdata.get("OPENAI_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
    raise RuntimeError("Ouvrez Secrets, ajoutez OPENAI_API_KEY et activez Accès au notebook, puis relancez cette cellule.") from None

# La clé est transmise en mémoire au processus isolé ; aucun fichier .env n'est créé.
try:
    session = client.ColabSession(PYTHON_RAG, PROJET, api_key=cle)
finally:
    del cle
materiel = session.request("status")
if not materiel["cuda_available"]:
    session.close()
    raise RuntimeError("Aucun GPU utilisable. Choisissez un GPU dans Exécution > Modifier le type d'exécution, puis reprenez à la cellule 1.")
print("GPU détecté :", materiel["cuda_device"])
print("Docling est configuré pour le GPU. Les embeddings BGE-M3 restent sur CPU, comme dans le compagnon.")
DONNEES = PROJET / "data/colab-30-pages"
connexion = session.request("hello", data_dir=str(DONNEES), model=modele, thread_id="verification-connexion")
display(Markdown(connexion["answer"]))
connexion_verifiee = True
conversation_id = uuid.uuid4().hex
resultat = None
print("Connexion OpenAI vérifiée. Passez à la cellule 3.")


## Le document de l'essai

Nous utilisons l'[appendice technique de Retraite Québec sur le calcul des rentes de Pierre et de Marie](https://www.retraitequebec.gouv.qc.ca/sites/default/files/SiteCollectionDocuments/RetraiteQuebec/fr/publications/nos-programmes/regime-de-rentes/consultation-publique/cp_etude_impact_part2.pdf). Ses exemples datent de **2009** ; ils ne décrivent pas les règles actuelles du régime.

Le PDF a 30 pages physiques, numérotées 53 à 82 dans le document imprimé. Les pages physiques 2, 4 et 14 sont blanches. Les citations du RAG indiquent les **pages physiques**.

Cette étape télécharge les modèles au premier lancement, puis analyse et indexe le PDF. Elle peut prendre plusieurs minutes. Laissez la cellule travailler jusqu'à **Document prêt**. Si vous l'arrêtez, les étapes déjà sauvegardées pourront être reprises ; un parsing non terminé doit recommencer.

Pour essayer ensuite un autre petit PDF numérique public, remplacez **url_pdf** et **titre_pdf** dans la cellule 3, puis posez une question sur ce nouveau document. Les trois réponses de référence plus bas concernent uniquement l’exemple de Retraite Québec. La base du document choisi est séparée de vos autres essais.


In [ ]:
# @title 3. Préparer le PDF
url_pdf = "https://www.retraitequebec.gouv.qc.ca/sites/default/files/SiteCollectionDocuments/RetraiteQuebec/fr/publications/nos-programmes/regime-de-rentes/consultation-publique/cp_etude_impact_part2.pdf" # @param {type:"string"}
titre_pdf = "Calcul des rentes de Pierre et de Marie, appendice technique de 2009" # @param {type:"string"}
import hashlib

document_pret = False
resultat = None
if not globals().get("connexion_verifiee"):
    raise RuntimeError("Exécutez d'abord les cellules 1 et 2 jusqu'au message Connexion OpenAI vérifiée.")
# Une base par URL garde le document choisi séparé des essais précédents.
DONNEES = PROJET / "data" / ("colab-" + hashlib.sha256(url_pdf.encode()).hexdigest()[:12])
bilan = session.request("ingest", source=url_pdf, title=titre_pdf, data_dir=str(DONNEES))
URL_EXEMPLE = "https://www.retraitequebec.gouv.qc.ca/sites/default/files/SiteCollectionDocuments/RetraiteQuebec/fr/publications/nos-programmes/regime-de-rentes/consultation-publique/cp_etude_impact_part2.pdf"
if url_pdf == URL_EXEMPLE:
    if bilan["record"]["document_id"] != "61417ac1abfe7571938f4dc26d5ba88db0b31f996770e68edb26ffc9f0d0888a":
        raise RuntimeError("Le fichier publié a changé. Les réponses de référence doivent être revérifiées.")
    pages_absentes = (set(range(1, 31)) - {2, 4, 14}) - set(bilan["covered_pages"])
    if bilan["pages"] != 30 or pages_absentes:
        raise RuntimeError(f"Extraction à vérifier : {bilan['pages']} pages déclarées, pages utiles sans provenance : {sorted(pages_absentes)}.")
if not bilan["manifest_ok"]:
    raise RuntimeError("L'ingestion n'est pas entièrement disponible. Ne passez pas encore aux questions.")
conversation_id = uuid.uuid4().hex
document_pret = True
print("Document prêt. Passez à la cellule 4.")
print(f"{bilan['pages']} pages ; {bilan['record']['parents']} parents ; {bilan['record']['children']} enfants.")
print(f"{bilan['tables']} tableaux ; {bilan['formulas']} formules ; {bilan['pictures']} figures.")
print("Dimensions des vecteurs :", bilan["vector_shape"])
print(f"Durée observée : {bilan['elapsed_seconds'] / 60:.1f} minutes.")
print("Ces contrôles vérifient la présence des données, pas l'exactitude de chaque extraction.")


## Posez votre première question

La question proposée porte sur une condition précise du document. Cliquez sur ▶, puis lisez la réponse et ouvrez ses citations.

Ensuite, remplacez simplement le texte du champ **question** et cliquez de nouveau sur ▶. Pour une relance comme « Et quelles sont ses limites ? », conservez la même conversation. La mémoire garde les trois derniers messages, questions et réponses comprises. Cochez **nouvelle_conversation** pour tester une question indépendante.

Pour vérifier d'autres aspects du PDF, vous pouvez copier l'une de ces questions :

- **Lire un tableau** : Dans l'exemple où Pierre décède lorsque Marie a 50 ans, quels gains annuels le tableau attribue-t-il à Marie de 23 à 31 ans après le transfert, et comment ce montant est-il obtenu ?
- **Comparer deux cas** : Lorsque Pierre décède à 65 ans et que Marie a également 65 ans, comment les pistes de solutions modifient-elles la rente totale mensuelle de Marie par rapport au régime de 2008, selon que les deux conjoints ont demandé leur retraite à 60 ans ou à 62 ans ?
- **Tester l'absence de preuve** : Quel est le numéro de téléphone personnel de Pierre ?


In [ ]:
# @title 4. Poser une question
question = "Selon les pistes de solutions décrites pour le décès de Pierre lorsque Marie a 50 ans, quelle proportion de ses gains peut être transférée à Marie, sur quelles années, et quelle limite s'applique à ce transfert ?" # @param {type:"string"}
nouvelle_conversation = False # @param {type:"boolean"}
resultat = None
if not globals().get("document_pret"):
    raise RuntimeError("Préparez d'abord le document avec la cellule 3.")
if not question.strip():
    raise RuntimeError("Écrivez une question dans le champ avant de lancer la cellule.")
if nouvelle_conversation:
    conversation_id = uuid.uuid4().hex
resultat = session.request("ask", question=question, thread_id=conversation_id)
display(Markdown(resultat["answer"]))
print("Branche choisie :", resultat["route"])
print("Parents demandés :", resultat["context_k"])
print("Question utilisée :", resultat["search_question"])


<details>
<summary><strong>Comparer les réponses aux pages du PDF</strong></summary>

Ces repères sont destinés au lecteur ; ils ne sont pas transmis au modèle.

**Condition du transfert, page physique 20 (imprimée 72).** Le transfert porte sur **60 % des gains de Pierre**, pour chaque année de vie commune. Le cumul avec les gains de Marie est limité au maximum assurable de l'année. Ne pas confondre ce transfert avec une rente temporaire.

**Lecture du tableau, page physique 21 (imprimée 73).** **36 516 dollars de gains annuels = 21 230 + 15 286 dollars**. Le transfert vaut 60 % de 25 476 dollars, arrondi au dollar. Il ne s'agit pas d'une pension mensuelle.

**Comparaison, pages physiques 27 à 30 (imprimées 79 à 82).** Départ à 60 ans : **577 → 559 dollars**, soit **18 dollars de moins par mois**. Départ à 62 ans : **605 → 663 dollars**, soit **58 dollars de plus par mois**. Ces montants concernent les exemples historiques du PDF.

Une réponse est à examiner même si ses liens sont valides. Vérifiez les chiffres, leurs unités, leurs conditions et les pages citées. Ces quelques essais ne constituent pas un benchmark actuariel.

</details>


## Facultatif, voir ce que le modèle a reçu

La cellule suivante affiche le texte et les mêmes fichiers image que ceux des parents envoyés au modèle, puis les premiers résultats des deux recherches. Elle n'effectue aucun nouvel appel API. Après une réponse directe, elle explique simplement qu'aucun document n'a été recherché.


In [ ]:
# @title 5. Voir les passages et les classements
if not globals().get("resultat"):
    print("Posez d'abord une question avec la cellule 4.")
elif resultat["route"] != "retrieve":
    print("Réponse directe : aucun passage documentaire n'a été envoyé au modèle.")
else:
    from IPython.display import Image
    for numero, parent in enumerate(resultat["context"]["parents"], start=1):
        display(Markdown(f"### Passage {numero}\n\n{parent['text']}"))
        # Les fichiers résident dans la même machine Colab que le processus RAG.
        for figure in parent["images"]:
            display(Image(filename=figure["path"]))
    recherche = resultat["retrieval"]
    print("BM25, cinq premiers parents :", recherche.get("bm25_hits", [])[:5])
    print("Recherche dense, cinq premiers enfants :", recherche.get("dense_hits", [])[:5])
    print("Parents retenus :", recherche.get("parent_ids", []))


## Facultatif, garder les fichiers du test

Votre copie du notebook reste dans Drive, mais **la machine Colab et son dossier de données sont temporaires**. La cellule 6 télécharge une archive avec le PDF, le parsing, les images, les index et la dernière réponse disponible. Elle n'inclut ni clé API, ni environnement Python, ni poids des modèles.

Pour un nouvel essai dans une machine Colab neuve, reprenez à la cellule 1. Ce notebook ne restaure pas automatiquement l'archive. Pour un essai dans la même session, relancer la cellule 3 réutilise les étapes complètes après vérification du PDF ; cette vérification retélécharge la source.


In [ ]:
# @title 6. Télécharger les fichiers du test
import json
import shutil
from google.colab import files

if "DONNEES" not in globals() or not (DONNEES / "manifest.json").is_file():
    raise RuntimeError("Il n'y a pas encore de document terminé à exporter. Exécutez la cellule 3.")
if globals().get("resultat"):
    (DONNEES / "derniere-reponse.json").write_text(
        json.dumps(resultat, ensure_ascii=False, indent=2), encoding="utf-8"
    )
archive = shutil.make_archive("/content/essai-rag-30-pages", "zip", root_dir=DONNEES)
files.download(archive)


## Si une cellule s'arrête

- **Secret introuvable ou accès refusé** : ouvrez Secrets, vérifiez le nom OPENAI_API_KEY et activez l'accès au notebook, puis relancez la cellule 2.
- **Clé, quota ou modèle refusé par OpenAI** : vérifiez l'accès API sur votre compte ou le nom du modèle dans la cellule 2. La clé d'un autre lecteur n'est jamais fournie par ce dépôt.
- **GPU indisponible** : consultez Exécution → Modifier le type d'exécution. Si Colab ne vous attribue pas de GPU, réessayez lorsque vous en disposez ; ce parcours ne garantit ni accès gratuit ni durée.
- **Calcul interrompu** : relancez la cellule 2, puis la cellule 3. Les étapes enregistrées restent réutilisables dans la même machine, mais la conversation est réinitialisée.
- **Autre erreur** : arrêtez-vous à cette cellule et conservez son message. Un message d'installation réussie ne prouve pas une ingestion réussie.

Le code métier reste celui du [compagnon GitHub](https://github.com/FranckTbn/docling-hybrid-rag) : Docling, parents et enfants, BM25 et BGE-M3, puis RRF et génération sourcée. Colab fournit l'environnement d'exécution et les champs pour l'essayer.

[Documentation Colab sur les ressources et les fichiers](https://research.google.com/colaboratory/faq.html) · [Code du client](https://github.com/FranckTbn/docling-hybrid-rag/blob/codex/colab-reader/colab_support.py) · [Code du processus RAG](https://github.com/FranckTbn/docling-hybrid-rag/blob/codex/colab-reader/lib/colab_worker.py)
